# **1. Setup — clone the repo and install dependencies**
Clones this project from GitHub and installs everything it needs (PDF/DOCX parsing, embeddings, reranking, the LLM, FastAPI, etc.).

In [1]:
!git clone https://github.com/sravanthskr/document-rag.git /content/document-rag
%cd /content/document-rag
!pip install -q -r requirements.txt
!apt-get install -y tesseract-ocr -q

Cloning into '/content/document-rag'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 60 (delta 10), reused 57 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 28.32 KiB | 5.66 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/document-rag
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 63.8 MB/s eta 0:00

# **2. Mount Google Drive**
Used for persistent storage — your uploaded documents, the vector database, and the document registry survive across Colab sessions instead of being wiped when the runtime disconnects.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **3. Launch the backend server**
Starts the FastAPI server in the background. It wraps the RAG pipeline (retrieval, reranking, generation) as REST endpoints and serves the web frontend.

In [3]:
%cd /content/document-rag
!nohup uvicorn app.api.server:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &

/content/document-rag


# **4. Wait for the server to be ready**
Models load lazily on first use, but the server itself needs a few seconds to start. If this prints "Not ready yet," just re-run this cell after a moment.

In [6]:
import time
time.sleep(20)
import requests
try:
    r = requests.get("http://localhost:8000/api/documents")
    print("Server is up:", r.status_code)
except Exception as e:
    print("Not ready yet, wait a bit and rerun this cell:", e)

Server is up: 200


# **5. Get a public URL**
Creates a public link (via ngrok) so the app is reachable in a browser, not just inside Colab. Requires a free ngrok account — get your authtoken at https://dashboard.ngrok.com/get-started/your-authtoken

In [7]:
from pyngrok import ngrok
import getpass

authtoken = getpass.getpass("Paste your ngrok authtoken: ")
ngrok.set_auth_token(authtoken)
ngrok.kill()
public_url = ngrok.connect(8000)
print("Open this URL:", public_url)

Paste your ngrok authtoken: ··········
Open this URL: NgrokTunnel: "https://protrude-nervy-quickstep.ngrok-free.dev" -> "http://localhost:8000"
